# Cart Pole
This environment is part of the **Classic Control** environments, which contain general information about the environment.

## Environment Specifications

- **Action Space:** `Discrete(2)`
- **Observation Space:** `Box([-4.8000002e+00, -3.4028235e+38, -4.1887903e-01, -3.4028235e+38], [4.8000002e+00, 3.4028235e+38, 4.1887903e-01, 3.4028235e+38], (4,), float32)`
- **Import:**

```python
gymnasium.make("CartPole-v1")
```

## Description

This environment corresponds to the version of the cart-pole problem described by Barto, Sutton, and Anderson in *"Neuronlike Adaptive Elements That Can Solve Difficult Learning Control Problem"*.

A pole is attached to a cart by an unactuated joint. The cart moves along a frictionless track, and the pole is initially positioned upright on the cart. The goal is to balance the pole by applying forces to move the cart left or right.

## Action Space

The action is a discrete value indicating the direction of the fixed force applied to the cart.

| Action | Description |
|:---:|---|
| `0` | Push the cart to the left |
| `1` | Push the cart to the right |

**Note:** The change in velocity caused by the applied force depends on the pole's angle. The pole's center of gravity affects the amount of energy required to move the cart underneath it.

## Observation Space

The observation is a NumPy array with shape `(4,)`, containing the following positions and velocities:

| Index | Observation | Minimum | Maximum |
|:---:|---|---:|---:|
| `0` | Cart Position | `-4.8` | `4.8` |
| `1` | Cart Velocity | `-Inf` | `Inf` |
| `2` | Pole Angle | Approximately `-0.418 rad` (`-24°`) | Approximately `0.418 rad` (`24°`) |
| `3` | Pole Angular Velocity | `-Inf` | `Inf` |

**Note:** The observation-space ranges above describe the possible values for each element, but they do not represent the allowed state values throughout an unterminated episode.

In particular:

- **Cart position:** The cart's x-position (index `0`) can take values between `-4.8` and `4.8`, but the episode terminates if the cart leaves the `(-2.4, 2.4)` range.
- **Pole angle:** The pole angle can be observed between approximately `-0.418` and `0.418` radians (±24°), but the episode terminates if the pole angle leaves the `(-0.2095, 0.2095)` range (±12°).

## Rewards

Since the goal is to keep the pole upright for as long as possible, the agent receives a reward of **+1 for every step**, including the termination step.

The reward threshold for `CartPole-v1` is **475**.

## Starting State

All observations are initialized with uniformly random values between `-0.05` and `0.05`.

## Episode End

An episode ends when any of the following conditions occurs:

### Termination

- **Pole Angle:** The pole angle exceeds ±12°.
- **Cart Position:** The cart's position exceeds ±2.4, meaning the cart's center reaches the edge of the display.

### Truncation

- **Episode Length:** The episode exceeds 500 steps in `CartPole-v1` (200 steps in `CartPole-v0`).

In [68]:
import numpy as np
import gymnasium as gym
from pathlib import Path
from collections import defaultdict
print("All Imports are successful")

All Imports are successful


## Solution using Q-learning table

In [69]:
from collections import defaultdict
import gymnasium as gym
import numpy as np
from gymnasium.spaces import Discrete


class CartPoleQLearningAgent:
	def __init__(
		self,
		env: gym.Env,
		learning_rate: float = 0.1,
		discount_factor: float = 0.99,
		initial_epsilon: float = 1.0,
		min_epsilon: float = 0.01,
		epsilon_decay: float = 0.995,
	) -> None:
		action_space = env.action_space

		if not isinstance(action_space, Discrete):
			raise TypeError("Expected a discrete action space.")

		self.env = env
		self.action_space = action_space

		self.learning_rate = learning_rate
		self.discount_factor = discount_factor
		self.epsilon = initial_epsilon
		self.min_epsilon = min_epsilon
		self.epsilon_decay = epsilon_decay

		self.bins: tuple[np.ndarray, ...] = (
			np.linspace(-4.8, 4.8, 100),
			np.linspace(-4.0, 4.0, 100),
			np.linspace(-0.418, 0.418, 100),
			np.linspace(-4.0, 4.0, 100),
		)

		self.q_table: defaultdict[tuple[int, ...], np.ndarray] = (
			defaultdict(
				lambda: np.zeros(
					self.action_space.n,
					dtype=np.float64,
				)
			)
		)

		self.errors: list[float] = []

	def discretize_state(
		self,
		state: np.ndarray,
	) -> tuple[int, ...]:
		return tuple(
			int(np.digitize(value, bin_edges))
			for value, bin_edges in zip(state, self.bins)
		)

	def choose_action(
		self,
		state: tuple[int, ...],
	) -> int:
		if np.random.random() < self.epsilon:
			return int(self.action_space.sample())

		return int(np.argmax(self.q_table[state]))

	def decay_epsilon(self) -> None:
		self.epsilon = max(
			self.min_epsilon,
			self.epsilon * self.epsilon_decay,
		)

	def update_q_value(
		self,
		state: tuple[int, ...],
		action: int,
		reward: float,
		terminated: bool,
		next_state: tuple[int, ...],
	) -> None:	
		best_next_action_value = (not terminated) * float(
			np.max(self.q_table[next_state])
		)

		td_target = (
			reward
			+ self.discount_factor * best_next_action_value
		)

		td_delta = td_target - self.q_table[state][action]

		self.q_table[state][action] += (
			self.learning_rate * td_delta
		)

		self.errors.append(float(td_delta))

In [70]:
import gymnasium as gym
show_env = False

env = gym.make("CartPole-v1",render_mode="human" if show_env else None)

EARLY_EXIT_REWARD = 1000
EARLY_EXIT_EPISODE = 10

episodes = 50000000
learning_rate = 0.1
discount_factor = 0.99
initial_epsilon = 1.0
min_epsilon = 0.05
epsilon_decay = 0.9999
episode_rewards: list[float] = []


cartpole_agent = CartPoleQLearningAgent(
  env=env,
  learning_rate=learning_rate,
  discount_factor=discount_factor,
  initial_epsilon=initial_epsilon,
  min_epsilon=min_epsilon,
  epsilon_decay=epsilon_decay
)
print(f"Training for {episodes} episodes with learning rate: \n"
      f"{cartpole_agent.learning_rate}, discount factor: {cartpole_agent.discount_factor},\n"
	  f" initial epsilon: {cartpole_agent.epsilon}, min epsilon: {cartpole_agent.min_epsilon},"
	  f" epsilon decay: {cartpole_agent.epsilon_decay}")

i = 0
reward_tracker = 0.0
total_thresh_passed_reward_count = 0

while i < episodes:
	i += 1
	state, info = env.reset()
	state = cartpole_agent.discretize_state(state)

	terminated = False
	truncated = False
	total_reward = 0.0
	reward_tracker = 0.0

	while not (terminated):
		action = cartpole_agent.choose_action(state)

		next_state, reward, terminated, truncated, info = (
			env.step(action)
		)

		next_state = cartpole_agent.discretize_state(next_state)
		reward = float(reward)

		cartpole_agent.update_q_value(
			state,
			action,
			reward,
			terminated,
			next_state,
		)
		state = next_state
		total_reward += reward
		
	reward_tracker = total_reward
	if reward_tracker >= 1000:
		total_thresh_passed_reward_count += 1
		print(f"Episode {i+1} finished with reward: {reward_tracker},total passed episodes: {total_thresh_passed_reward_count}")
		if total_thresh_passed_reward_count >= EARLY_EXIT_EPISODE:
			print(F"Early exiting at expisode {i}, since , we got {total_thresh_passed_reward_count} total passed episodes with reward >= {EARLY_EXIT_REWARD}")
			break

	cartpole_agent.decay_epsilon()

	episode_rewards.append(total_reward)


	is_printable = ((i+1) % 10000 == 0) 
	if is_printable:
		print(f"Episode {i} finished with epsilon: {cartpole_agent.epsilon}, average reward: "
			f"{np.mean(episode_rewards[-1000:])},min reward: {np.min(episode_rewards[-1000:])},"
			f" max reward: {np.max(episode_rewards[-1000:])}")
	
print(f"Average reward over {i} episodes: {np.mean(episode_rewards)}, {len(episode_rewards)}")
env.close()

Training for 50000000 episodes with learning rate: 
0.1, discount factor: 0.99,
 initial epsilon: 1.0, min epsilon: 0.05, epsilon decay: 0.9999
Episode 9999 finished with epsilon: 0.36789783621659083, average reward: 45.97,min reward: 9.0, max reward: 167.0
Episode 19999 finished with epsilon: 0.13533528301106, average reward: 114.228,min reward: 13.0, max reward: 592.0
Episode 21401 finished with reward: 1077.0, consecutive episodes: 1
Episode 22871 finished with reward: 1174.0, consecutive episodes: 2
Episode 22946 finished with reward: 1979.0, consecutive episodes: 3
Episode 22959 finished with reward: 1791.0, consecutive episodes: 4
Episode 23011 finished with reward: 1186.0, consecutive episodes: 5
Episode 23024 finished with reward: 1008.0, consecutive episodes: 6
Episode 23802 finished with reward: 1091.0, consecutive episodes: 7
Episode 23889 finished with reward: 1718.0, consecutive episodes: 8
Episode 24149 finished with reward: 1981.0, consecutive episodes: 9
Episode 24471 f